# Feature Engineering for Community Area Aggregation
This notebook works as the other Feature Engineering Notebooks except it uses the dataset aggregated on Community Area

In [2]:
import pandas as pd
from pathlib import Path
import numpy as np
import holidays
from sklearn.metrics.pairwise import haversine_distances
import gc

In [2]:
# Load all complete grids from the data folder
data_folder = Path("../data/processed/")
grid_files = list(data_folder.glob("complete_grid_*.parquet"))

# Removed any grid files that contain "h3_8" in their name
grid_files = [f for f in grid_files if "h3_8" not in f.stem]
print(f"Found {len(grid_files)} complete grid files.")

# load each grid file into own data frame and save them in a dictionary
grid_dfs = {}
for grid_file in grid_files:
    grid_name = grid_file.stem  # Get the file name without extension
    grid_dfs[grid_name] = pd.read_parquet(grid_file)
    print(f"Loaded {grid_name} with shape: {grid_dfs[grid_name].shape}")

Found 6 complete grid files.
Loaded complete_grid_community_daily with shape: (66451, 14)
Loaded complete_grid_h3_7_hourly with shape: (2236896, 14)
Loaded complete_grid_community_hourly with shape: (1594824, 14)
Loaded complete_grid_h3_7_daily with shape: (93204, 14)
Loaded complete_grid_census_hourly with shape: (10273152, 14)
Loaded complete_grid_census_daily with shape: (428048, 14)


In [3]:
# force the key dtypes as string
poi_keys = {"geoid10": str, "commarea": str, "h3_index": str}

# Load the POI category wide files and weather data
pois_cat_wide_7 = pd.read_csv("../data/processed/chicago_pois_category_wide_7.csv", dtype=poi_keys)
pois_cat_wide_census = pd.read_csv("../data/processed/chicago_pois_category_wide_census.csv", dtype=poi_keys)
pois_cat_wide_comm = pd.read_csv("../data/processed/chicago_pois_category_wide_community.csv", dtype=poi_keys)
weather_df = pd.read_parquet("../data/weather_data.parquet")

for name, df in [("pois_7", pois_cat_wide_7),
                 ("pois_census", pois_cat_wide_census), ("pois_community", pois_cat_wide_comm)]:
    print(f"Loaded {name:15s} shape: {str(df.shape):10s} key: {df.columns[0]}")
print(f"Loaded weather data with shape: {weather_df.shape}")

Loaded pois_7          shape: (152, 15)  key: h3_index
Loaded pois_census     shape: (797, 15)  key: geoid10
Loaded pois_community  shape: (77, 15)   key: commarea
Loaded weather data with shape: (20712, 9)


In [4]:
# Print the column names of each grid data frame
for grid_name, grid_df in grid_dfs.items():
    print(f"Columns names in {grid_name}:")
    print(grid_df.columns)

Columns names in complete_grid_community_daily:
Index(['timestamp', 'commarea', 'Total_Trip_Start', 'Unique Taxis',
       'AvgTripSeconds', 'AvgTripMiles', 'AvgFare', 'MostCommonCompany',
       'CompanyCount', 'PickupLongitude', 'PickupLatitude', 'Total_Trip_End',
       'lat', 'lon'],
      dtype='str')
Columns names in complete_grid_h3_7_hourly:
Index(['timestamp', 'h3_index_7', 'Total_Trip_Start', 'Unique Taxis',
       'AvgTripSeconds', 'AvgTripMiles', 'AvgFare', 'MostCommonCompany',
       'CompanyCount', 'PickupLongitude', 'PickupLatitude', 'Total_Trip_End',
       'lat', 'lon'],
      dtype='str')
Columns names in complete_grid_community_hourly:
Index(['timestamp', 'commarea', 'Total_Trip_Start', 'Unique Taxis',
       'AvgTripSeconds', 'AvgTripMiles', 'AvgFare', 'MostCommonCompany',
       'CompanyCount', 'PickupLongitude', 'PickupLatitude', 'Total_Trip_End',
       'lat', 'lon'],
      dtype='str')
Columns names in complete_grid_h3_7_daily:
Index(['timestamp', 'h3_index_7', 

In [5]:
# Find Null Values and add percentage of NaN values for each column
# Only print columns with NaN values
def generate_nan_report(df):
    is_na_df = df.isna().sum()
    is_na_df = is_na_df[is_na_df > 0]
    is_na_df = pd.DataFrame(is_na_df, columns=['NaN Count'])
    is_na_df['Total Count'] = len(df)
    nan_percentage = (is_na_df['NaN Count'] / is_na_df['Total Count']) * 100
    # Format the NaN percentage as a string with 2 decimal places and add a percentage sign
    is_na_df['NaN Percentage'] = nan_percentage.map(lambda x: f"{x:.2f}%")
    return is_na_df

# Check for any NaN values in all columns
print("Only showing columns with NaN values:")
for grid_name, grid_df in grid_dfs.items():
    nan_report = generate_nan_report(grid_df)
    if not nan_report.empty:
        print(f"NaN Report for {grid_name}:")
        print(nan_report)
        print(40 * "-")
    else:
        print(f"No NaN values found in {grid_name}.")
        print(40 * "-")

Only showing columns with NaN values:
NaN Report for complete_grid_community_daily:
                NaN Count  Total Count NaN Percentage
AvgTripSeconds        609        66451          0.92%
AvgTripMiles          609        66451          0.92%
AvgFare               609        66451          0.92%
----------------------------------------
NaN Report for complete_grid_h3_7_hourly:
                NaN Count  Total Count NaN Percentage
AvgTripSeconds    2075923      2236896         92.80%
AvgTripMiles      2075923      2236896         92.80%
AvgFare           2075923      2236896         92.80%
----------------------------------------
NaN Report for complete_grid_community_hourly:
                NaN Count  Total Count NaN Percentage
AvgTripSeconds     720490      1594824         45.18%
AvgTripMiles       720490      1594824         45.18%
AvgFare            720490      1594824         45.18%
----------------------------------------
NaN Report for complete_grid_h3_7_daily:
               

## 1. Spatial Features

In [6]:
# Calculate distance to Chicago Loop (downtown center)
# The Haversine distance is the shortest route over the earth's surface
# (distance between two points on a sphere, calculated using their latitudes and longitudes)
def new_haversine_distance_to_loop(lat1, lon1, ):
    # Use scikit-learn's haversine_distances function
    lat2=41.8781 # Chicago Loop latitude
    lon2=-87.6298 # Chicago Loop longitude
    # Convert latitudes and longitudes from degrees to radians
    coords_1 = np.radians(np.column_stack((lat1, lon1)))
    coords_2 = np.radians(np.array([[lat2, lon2]]))
    # Convert from radians to kilometers
    distances = haversine_distances(coords_1, coords_2) * 6371  
    return distances.flatten()

## 2. Time Features

In [7]:
def add_holiday_feature(grid_df, temporal_column):
    dates = grid_df[temporal_column].dt.normalize()
    us_holidays = holidays.USA(years=dates.dt.year.unique(), state='IL')
    holiday_dates = pd.DatetimeIndex(sorted(us_holidays.keys()))

    grid_df['is_holiday'] = dates.isin(holiday_dates).astype(int)

    # "Near a holiday" means the calendar day before or after one.
    day = pd.Timedelta(days=1)
    near = holiday_dates.union(holiday_dates - day).union(holiday_dates + day)
    grid_df['is_near_holiday'] = dates.isin(near).astype(int)
    return grid_df

def temporal_feature_engineering(grid_df, temporal_column, temporal_level):
    # The hour features only exist for the hourly grids, not in daily grids
    # Calendar features
    if temporal_level == 'hourly':
        grid_df['hour_of_day'] = grid_df[temporal_column].dt.hour
    grid_df['month'] = grid_df[temporal_column].dt.month
    grid_df['day_of_week'] = grid_df[temporal_column].dt.weekday
    grid_df['is_weekend'] = (grid_df['day_of_week'] >= 5).astype(int)
    grid_df['weekday'] = grid_df[temporal_column].dt.weekday
    grid_df["season"] = grid_df[temporal_column].dt.month % 12 // 3 + 1 # Mapping: 1 = winter, 2 = spring, 3 = summer, 4 = autumn

    # Cyclic temporal features
    if temporal_level == 'hourly':
        grid_df['hour_sin'] = np.sin(2 * np.pi * grid_df['hour_of_day'] / 24)
        grid_df['hour_cos'] = np.cos(2 * np.pi * grid_df['hour_of_day'] / 24)
    grid_df['month_sin'] = np.sin(2 * np.pi * grid_df['month'] / 12)
    grid_df['month_cos'] = np.cos(2 * np.pi * grid_df['month'] / 12)
    grid_df["season_sin"] = np.sin(2 * np.pi * grid_df["season"] / 4)
    grid_df["season_cos"] = np.cos(2 * np.pi * grid_df["season"] / 4)
    return grid_df

# Historical base demand — computed ONLY from train (no leakage)
def add_base_demand(df, base, keys, global_base):
    df = df.merge(base, on=keys, how="left")
    # unseen keys -> global mean
    df["base_demand"] = df["base_demand"].fillna(global_base)  
    return df

def base_demand_keys(spatial_column, temporal_level):
    """The profile a cell's historical demand is averaged over, per temporal level."""
    if temporal_level == 'hourly':
        return [spatial_column, 'hour_of_day', 'month', 'day_of_week']
    # A daily grid has no hour; day_of_week already carries the weekday/weekend distinction.
    return [spatial_column, 'day_of_week', 'month']

## 3. Weather Features

In [8]:
def create_weather_features(grid_df, weather_df, temporal_column, time_step_column="time_step"):

    weather_df = weather_df.copy()

    # combine the raw wind data into the wind speed and direction
    weather_df["wind_speed"] = (weather_df["u10"]**2 + weather_df["v10"]**2) ** 0.5
    weather_df["wind_dir"] = (270 - np.degrees(np.arctan2(weather_df["u10"], weather_df["v10"]))*180/np.pi)%360

    weather_columns = [c for c in weather_df.columns if c != time_step_column]

    # join weather and taxi data upon the time as key 
    grid_df = grid_df.merge(
        weather_df,
        left_on=temporal_column,
        right_on=time_step_column,
        how="left"
    )

    # Check that the join worked and that there are no missing weather values in the grid
    missing = int(grid_df[weather_columns].isna().any(axis=1).sum())
    assert missing == 0, (
        f"{missing:,} rows have no weather -- the grid extends past the weather series. "
        f"Re-run 1.4, which cuts the taxi data at the weather cutoff."
    )

    # The join key just duplicates `timestamp`, so it is not a feature.
    grid_df = grid_df.drop(columns=[time_step_column])

    return grid_df

## 4. POI Features

In [9]:
# Map the spatial level to the corresponding POI data frame and the column to merge on
POI_LEVELS = {
    'h3_7':      {'spatial_column': 'h3_index_7', 'pois': pois_cat_wide_7},
    'census':    {'spatial_column': 'geoid10',    'pois': pois_cat_wide_census},
    'community': {'spatial_column': 'commarea',   'pois': pois_cat_wide_comm},
}

# Rename the spatial column to the grid's own key once, here, so a single merge works unchanged at every level.
for level, spec in POI_LEVELS.items():
    poi_key = 'h3_index' if level.startswith('h3') else spec['spatial_column']
    spec['pois'] = spec['pois'].rename(columns={poi_key: spec['spatial_column']})

def split_grid_name(grid_name):
    # complete_grid_h3_7_hourly' -> ('h3_7', 'hourly')
    spatial_level, temporal_level = grid_name.removeprefix('complete_grid_').rsplit('_', 1)
    return spatial_level, temporal_level

def add_poi_features(grid_df, grid_name):
    # Merge the POI table belonging to this grid's spatial level, resolved from the grid name.
    spatial_level, _ = split_grid_name(grid_name)
    spec = POI_LEVELS[spatial_level]
    pois, spatial_column = spec['pois'], spec['spatial_column']

    grid_df = grid_df.merge(pois, on=spatial_column, how='left')

    # Fill Na values in POI columns with 0.
    poi_columns = [c for c in pois.columns if c.startswith('poi_cat_')]
    grid_df[poi_columns] = grid_df[poi_columns].fillna(0).astype(int)

    return grid_df

for grid_name in grid_dfs:
    spatial_level, temporal_level = split_grid_name(grid_name)
    spec = POI_LEVELS[spatial_level]
    print(f"{grid_name:32s} -> {spatial_level:10s} {temporal_level:7s} "
          f"merges POIs on '{spec['spatial_column']}' ({len(spec['pois'])} units)")

complete_grid_community_daily    -> community  daily   merges POIs on 'commarea' (77 units)
complete_grid_h3_7_hourly        -> h3_7       hourly  merges POIs on 'h3_index_7' (152 units)
complete_grid_community_hourly   -> community  hourly  merges POIs on 'commarea' (77 units)
complete_grid_h3_7_daily         -> h3_7       daily   merges POIs on 'h3_index_7' (152 units)
complete_grid_census_hourly      -> census     hourly  merges POIs on 'geoid10' (797 units)
complete_grid_census_daily       -> census     daily   merges POIs on 'geoid10' (797 units)


# Feature Engineering Loop

In [10]:
DROP_COLUMNS = [
    'Unique Taxis', 'CompanyCount', 'MostCommonCompany',
    'AvgFare', 'AvgTripMiles', 'AvgTripSeconds',
    'Total_Trip_End',
    'PickupLatitude', 'PickupLongitude',
]

def engineer_features(grid_df, grid_name):
    """Add every feature block to one grid. The grid name resolves both the spatial and the
    temporal level, so nothing has to be passed in by hand."""
    spatial_level, temporal_level = split_grid_name(grid_name)
    temporal_column = 'timestamp'

    # Drop these columns as they are not needed for feature engineering and may cause data leakage
    grid_df = grid_df.drop(columns=DROP_COLUMNS, errors='ignore')

    grid_df['distance_to_loop'] = new_haversine_distance_to_loop(grid_df['lat'], grid_df['lon'])
    grid_df = add_holiday_feature(grid_df, temporal_column)
    grid_df = temporal_feature_engineering(grid_df, temporal_column, temporal_level)
    grid_df = create_weather_features(grid_df, weather_df, temporal_column)
    grid_df = add_poi_features(grid_df, grid_name)
    return grid_df

## 5. Prediction Data Split for Train, Validation and Test
**Validation Strategy**: Use a robust **Temporal Split** (avoiding autokorrelation and data leakage from random splits) to evaluate out-of-sample predictive performance.

In [11]:
def temporal_split(grid_df, grid_name):
    # Split one grid into train/val/test on time (50/20/30) and attach the training-set base demand.
    # Both the spatial key and the base-demand profile come from the grid name, so this works for every level. 
    # A random split would leak: consecutive hours in the same cell are highly autocorrelated.
    spatial_level, temporal_level = split_grid_name(grid_name)
    spatial_column = POI_LEVELS[spatial_level]['spatial_column']
    temporal_column = 'timestamp'

    unique_dates = pd.Series(grid_df[temporal_column].dt.date.unique()).sort_values().reset_index(drop=True)
    train_end = unique_dates.iloc[int(len(unique_dates) * 0.50)]
    val_end   = unique_dates.iloc[int(len(unique_dates) * 0.70)]

    dates = grid_df[temporal_column].dt.date
    df_train = grid_df[dates < train_end]
    df_val   = grid_df[(dates >= train_end) & (dates < val_end)]
    df_test  = grid_df[dates >= val_end]

    print(f"  train < {train_end} | val < {val_end} | "
          f"rows {len(df_train):,} / {len(df_val):,} / {len(df_test):,}")

    # Historical base demand, computed ONLY on train and applied to all three splits.
    keys = base_demand_keys(spatial_column, temporal_level)
    base = df_train.groupby(keys)["Total_Trip_Start"].mean().rename("base_demand").reset_index()
    global_base = df_train["Total_Trip_Start"].mean()
    print(f"  base_demand keyed on {keys}")

    return {
        'train': add_base_demand(df_train, base, keys, global_base),
        'val':   add_base_demand(df_val, base, keys, global_base),
        'test':  add_base_demand(df_test, base, keys, global_base),
    }


def export_grid_dfs(grid_name, grid_df, output_folder="../data/prediction_split/"):
    #Split `grid_df` and write it to prediction_split/<spatial_level>/<temporal_level>/<split>.parquet
    spatial_level, temporal_level = split_grid_name(grid_name)
    output_folder = Path(output_folder) / spatial_level / temporal_level
    output_folder.mkdir(parents=True, exist_ok=True)

    splits = temporal_split(grid_df, grid_name)
    for split_name, df in splits.items():
        df.to_parquet(output_folder / f"{split_name}.parquet", index=False)
    print(f"  exported train/val/test to {output_folder}")

In [12]:
# Engineer, split and export one grid at a time.
# Feature engineering roughly quadruples a grid's width, so holding all six engineered grids at once needs several GB, 
# Each grid is therefore reloaded from disk, processed, written and dropped before the next one starts.
del grid_dfs
gc.collect()

for grid_file in sorted(grid_files):
    grid_name = grid_file.stem
    spatial_level, temporal_level = split_grid_name(grid_name)
    print(f"{spatial_level} / {temporal_level}")

    grid_df = engineer_features(pd.read_parquet(grid_file), grid_name)
    print(f"  {grid_df.shape[0]:,} rows x {grid_df.shape[1]} columns after feature engineering")

    export_grid_dfs(grid_name, grid_df)

    del grid_df
    gc.collect()

print("\nAll grids engineered, split and exported.")

census / daily
  428,048 rows x 41 columns after feature engineering
  train < 2025-03-07 | val < 2025-08-27 | rows 213,776 / 85,808 / 128,464
  base_demand keyed on ['geoid10', 'day_of_week', 'month']
  exported train/val/test to ../data/prediction_split/census/daily
census / hourly
  10,273,152 rows x 44 columns after feature engineering
  train < 2025-03-07 | val < 2025-08-27 | rows 5,130,624 / 2,059,392 / 3,083,136
  base_demand keyed on ['geoid10', 'hour_of_day', 'month', 'day_of_week']
  exported train/val/test to ../data/prediction_split/census/hourly
community / daily
  66,451 rows x 41 columns after feature engineering
  train < 2025-03-07 | val < 2025-08-27 | rows 33,187 / 13,321 / 19,943
  base_demand keyed on ['commarea', 'day_of_week', 'month']
  exported train/val/test to ../data/prediction_split/community/daily
community / hourly
  1,594,824 rows x 44 columns after feature engineering
  train < 2025-03-07 | val < 2025-08-27 | rows 796,488 / 319,704 / 478,632
  base_deman

# Create Table for Report

In [ ]:
#| label: tbl-engineered-features
#| tbl-cap: "List of all Engineered Features."
from IPython.display import HTML
pd.set_option('display.max_colwidth', None)
table_features = pd.DataFrame([
    {"Engineered Feature": "distance_to_loop",
     "Description": "Great-circle (haversine) distance from each grid cell's coordinates to the Chicago Loop (downtown reference point), used as a proxy for centrality/urban density.",
     "DataType": "float64"},
    {"Engineered Feature": "wind_speed",
     "Description": "Magnitude of the horizontal wind vector, computed as sqrt(u10² + v10²) from the 10m u/v wind components.",
     "DataType": "float64"},
    {"Engineered Feature": "wind_dir",
     "Description": "Meteorological wind direction (direction the wind blows from), computed via atan2(u10, v10) and converted to compass degrees (0-360°).",
     "DataType": "float64"},
    {"Engineered Feature": "is_holiday",
     "Description": "Binary indicator flagging official U.S. federal holidays observed in Illinois, expected to shift ride-hailing demand relative to regular weekdays.",
     "DataType": "int64"},
    {"Engineered Feature": "is_near_holiday",
     "Description": "Binary indicator flagging the calendar day immediately before or after a public holiday, capturing anticipatory or spillover demand effects.",
     "DataType": "int64"},
    {"Engineered Feature": "hour_of_day",
     "Description": "Hour extracted from the timestamp, capturing intraday demand cycles. Only present in hourly-resolution grids.",
     "DataType": "int64"},
    {"Engineered Feature": "hour_sin",
     "Description": "Sine transformation of hour_of_day, encoding time of day on a continuous circular scale. Only present in hourly-resolution grids.",
     "DataType": "float64"},
    {"Engineered Feature": "hour_cos",
     "Description": "Cosine transformation of hour_of_day, paired with hour_sin to avoid the discontinuity between hour 23 and hour 0. Only present in hourly-resolution grids.",
     "DataType": "float64"},
    {"Engineered Feature": "month",
     "Description": "Calendar month extracted from the timestamp, capturing seasonal variation.",
     "DataType": "int64"},
    {"Engineered Feature": "month_sin",
     "Description": "Sine transformation of month, encoding the annual cycle continuously.",
     "DataType": "float64"},
    {"Engineered Feature": "month_cos",
     "Description": "Cosine transformation of month, paired with month_sin to avoid the discontinuity between December and January.",
     "DataType": "float64"},
    {"Engineered Feature": "day_of_week",
     "Description": "Numeric day of week extracted from the timestamp (0 = Monday, ..., 6 = Sunday), capturing weekly demand patterns.",
     "DataType": "int64"},
    {"Engineered Feature": "is_weekend",
     "Description": "Binary indicator flagging Saturday and Sunday (day_of_week >= 5), expected to correlate with different ride purposes (leisure vs. commute).",
     "DataType": "int64"},
    {"Engineered Feature": "weekday",
     "Description": "Numeric day of week extracted from the timestamp; identical in definition to day_of_week (0 = Monday, ..., 6 = Sunday).",
     "DataType": "int64"},
    {"Engineered Feature": "season",
     "Description": "Meteorological season derived from month, encoded as an integer (1 = Winter, 2 = Spring, 3 = Summer, 4 = Autumn), capturing broader seasonal demand and weather regimes.",
     "DataType": "int64"},
    {"Engineered Feature": "season_sin",
     "Description": "Sine transformation of season, representing the seasonal cycle continuously.",
     "DataType": "float64"},
    {"Engineered Feature": "season_cos",
     "Description": "Cosine transformation of season, paired with season_sin to avoid a discontinuity between the last and first season of the year.",
     "DataType": "float64"},
    {"Engineered Feature": "base_demand",
     "Description": "Historical average of Total_Trip_Start, computed on the training split only and grouped by spatial cell, month, day_of_week, and (for hourly grids) hour_of_day; unseen key combinations are filled with the global training mean.",
     "DataType": "float64"},
])

_colgroup = '<colgroup><col style="width:22%"><col style="width:63%"><col style="width:15%"></colgroup>'
HTML(table_features.to_html(escape=False, index=False).replace('<thead>', _colgroup + '<thead>', 1))


Engineered Feature,Description,DataType
distance_to_loop,"Great-circle (haversine) distance from each grid cell's coordinates to the Chicago Loop (downtown reference point), used as a proxy for centrality/urban density.",float64
wind_speed,"Magnitude of the horizontal wind vector, computed as sqrt(u10² + v10²) from the 10m u/v wind components.",float64
wind_dir,"Meteorological wind direction (direction the wind blows from), computed via atan2(u10, v10) and converted to compass degrees (0-360°).",float64
is_holiday,"Binary indicator flagging official U.S. federal holidays observed in Illinois, expected to shift ride-hailing demand relative to regular weekdays.",int64
is_near_holiday,"Binary indicator flagging the calendar day immediately before or after a public holiday, capturing anticipatory or spillover demand effects.",int64
hour_of_day,"Hour extracted from the timestamp, capturing intraday demand cycles. Only present in hourly-resolution grids.",int64
hour_sin,"Sine transformation of hour_of_day, encoding time of day on a continuous circular scale. Only present in hourly-resolution grids.",float64
hour_cos,"Cosine transformation of hour_of_day, paired with hour_sin to avoid the discontinuity between hour 23 and hour 0. Only present in hourly-resolution grids.",float64
month,"Calendar month extracted from the timestamp, capturing seasonal variation.",int64
month_sin,"Sine transformation of month, encoding the annual cycle continuously.",float64
